# 4e_dataset — Entraînement segmentation d'instances (Mask R-CNN)

Compte les symboles CFO/CFA/SSI sur les plans. Dataset auto-étiqueté depuis les DWG.

**Avant de lancer :** Exécution → *Modifier le type d'exécution* → **GPU** (T4 suffit).

**Étapes :** 1) installer → 2) uploader `coco_colab.zip` → 3) entraîner → 4) évaluer → 5) compter.

## 1. Installation (detectron2 + dépendances)

In [ ]:
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| detectron2', detectron2.__version__)

## 2. Uploader le dataset
Uploade le fichier **`coco_colab.zip`** (produit par `prepare_for_colab.py`).

In [ ]:
from google.colab import files
up = files.upload()            # choisis coco_colab.zip
!unzip -oq coco_colab.zip -d .
!ls coco_colab/train/images | head; echo ...; ls coco_colab/val/images | wc -l

## 3. Enregistrer le dataset COCO

In [ ]:
import json
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog

register_coco_instances('plans_train', {}, 'coco_colab/train/annotations.json', 'coco_colab/train/images')
register_coco_instances('plans_val',   {}, 'coco_colab/val/annotations.json',   'coco_colab/val/images')
NUM_CLASSES = len(json.load(open('coco_colab/train/annotations.json'))['categories'])
print('classes:', NUM_CLASSES)
print('train:', len(DatasetCatalog.get('plans_train')), 'tuiles |',
      'val:', len(DatasetCatalog.get('plans_val')), 'tuiles')

### Vérif visuelle des annotations (masques sur les symboles)

In [ ]:
import random, cv2
from detectron2.utils.visualizer import Visualizer
from google.colab.patches import cv2_imshow
md = MetadataCatalog.get('plans_train')
for d in random.sample(DatasetCatalog.get('plans_train'), 2):
    img = cv2.imread(d['file_name'])
    v = Visualizer(img[:, :, ::-1], metadata=md, scale=1.0)
    out = v.draw_dataset_dict(d)
    cv2_imshow(out.get_image()[:, :, ::-1])

## 4. Entraînement (Mask R-CNN R50-FPN, poids COCO pré-entraînés)
`MAX_ITER` est **auto-calculé** selon la taille du dataset (~40 époques). Surveille la
perte affichée : si elle descend encore à la fin, relance avec plus d'époques ;
si elle stagne tôt, tu peux réduire.

In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
import json

# cellule AUTONOME : reenregistre les datasets si le runtime a redemarre
# (sinon NameError: DatasetCatalog). register_* leve si deja enregistre.
for name, split in (('plans_train', 'train'), ('plans_val', 'val')):
    if name not in DatasetCatalog.list():
        register_coco_instances(name, {}, f'coco_colab/{split}/annotations.json',
                                f'coco_colab/{split}/images')
NUM_CLASSES = len(json.load(open('coco_colab/train/annotations.json'))['categories'])
print('classes:', NUM_CLASSES)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file('COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url('COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml')
cfg.DATASETS.TRAIN = ('plans_train',)
cfg.DATASETS.TEST = ('plans_val',)
cfg.DATALOADER.NUM_WORKERS = 2
# le dataset contient ~10% de tuiles de FOND (aucun symbole) : elles apprennent
# au modele a ne rien detecter sur le vide. detectron2 les jetterait par defaut.
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.001

# nombre d'iterations AUTO-ADAPTE a la taille du dataset (~40 epochs), min 2000
n_train = len(DatasetCatalog.get('plans_train'))
iters_per_epoch = max(1, n_train // cfg.SOLVER.IMS_PER_BATCH)
cfg.SOLVER.MAX_ITER = max(2000, iters_per_epoch * 40)
cfg.SOLVER.STEPS = (int(cfg.SOLVER.MAX_ITER * 0.7), int(cfg.SOLVER.MAX_ITER * 0.9))
print(f'{n_train} tuiles train -> MAX_ITER = {cfg.SOLVER.MAX_ITER}')

cfg.SOLVER.WARMUP_ITERS = min(1000, cfg.SOLVER.MAX_ITER // 10)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256

# ============================================================================
# LEVIERS DE REPLI — activer UN A LA FOIS si le run avec les nouvelles ancres ne
# suffit pas (garder l'attribution propre : un seul changement par run).
# Diagnostic 2026-07-11 (calibrate_thresholds.py sur le run precedent) : 2 modes
# d'echec au-dela de NOURRICE/SUSPENSION/WIFI (que les ancres corrigent) —
#   (a) CVC : symboles ~6x6 px, sous la plus petite ancre (8) -> jamais matches ;
#   (b) CAMERA & co : bonne taille mais detections sur les mauvaises images
#       = confusion de classe / faux positifs (les ancres n'y peuvent rien).
#
# Levier 1 ACTIF (2026-07-12) — sur-echantillonne les images contenant des classes
# rares/confondues (SUSPENSION, LAMPE_POSER, BORNE_WIFI...). L'utilisateur confirme
# que ces symboles sont visuellement DISTINCTS -> le modele DOIT pouvoir les separer,
# la confusion vient du desequilibre, pas d'une ambiguite de forme.
cfg.DATALOADER.SAMPLER_TRAIN = "RepeatFactorTrainingSampler"
cfg.DATALOADER.REPEAT_THRESHOLD = 0.30
#
# Levier 2 — boites dures a matcher (si NOURRICE reste faible malgre les ancres) :
#   abaisse le seuil RPN positif 0.7 -> 0.5 (plus d'ancres positives par objet).
# cfg.MODEL.RPN.IOU_THRESHOLDS = [0.3, 0.5]
#
# Levier 3 — symboles minuscules (CVC 6px) : ajouter une ancre plus fine au niveau
#   le plus fin. NB : passe a 12 ancres/pos -> surveiller l'OOM.
# cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[4, 8, 12], [16, 24], [32, 48], [64, 96], [128, 192]]
#   (ATTENTION : le nb d'ancres/pos doit etre EGAL a tous les niveaux pour la tete
#    RPN partagee — ci-dessus le 1er niveau a 3 tailles, les autres 2 : INVALIDE.
#    Pour 6px il vaut mieux RENDRE les plans a plus haute resolution en amont.)
# ============================================================================
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
# UNE taille PAR NIVEAU FPN (p2..p6). Une liste unique serait diffusee a tous les
# niveaux -> 5 tailles x 3 ratios = 15 ancres/position au lieu de 3, soit ~800k ancres
# sur une tuile 800px. pairwise_iou(ancres x verites) explose sur nos tuiles denses
# (jusqu'a 507 instances) -> CUDA OOM. Tailles petites : nos symboles font ~5-16 px.
# ANCRES SIMPLES (3/pos). Le run 2026-07-12 avec 10 ancres/pos (2 tailles x 5 ratios)
# N'A RIEN AMELIORE (NOURRICE recall 91->52, SUSPENSION/WIFI toujours 0) : le vrai
# goulot n'etait PAS les ancres mais la CONFUSION DE CLASSE (le modele voit le
# symbole, met la mauvaise etiquette) + un SPLIT qui affamait ces classes. Corrige
# cote donnees (split : classes concentrees -> train ; + RepeatFactorSampler
# ci-dessous). On revient donc aux ancres simples qui suffisent et coutent moins.
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[8], [16], [32], [64], [128]]  # symboles petits

# nos tuiles sont denses : on bride le nombre de propositions gardees a l'entrainement
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 4000     # defaut 12000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 1000    # defaut 2000
cfg.SOLVER.AMP.ENABLED = True               # demi-precision : ~2x moins de VRAM
cfg.TEST.DETECTIONS_PER_IMAGE = 400                        # plans tres denses
cfg.OUTPUT_DIR = 'output'
import os; os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

## 5. Évaluation (AP COCO sur le split val)
C'est ici le **vrai chiffre de viabilité** : performance sur des plans jamais vus.

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

cfg.MODEL.WEIGHTS = 'output/model_final.pth'
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
ev = COCOEvaluator('plans_val', output_dir='output')
loader = build_detection_test_loader(cfg, 'plans_val')
print(inference_on_dataset(trainer.model, loader, ev))

## 6. Comptage : détections par classe sur une image val vs vérité

In [ ]:
from detectron2.engine import DefaultPredictor
from collections import Counter
import numpy as np
pred = DefaultPredictor(cfg)
names = MetadataCatalog.get('plans_val').thing_classes
val = DatasetCatalog.get('plans_val')
for d in random.sample(val, min(3, len(val))):
    img = cv2.imread(d['file_name'])
    out = pred(img)['instances'].to('cpu')
    det = Counter(names[c] for c in out.pred_classes.numpy())
    gt  = Counter(names[a['category_id']] for a in d['annotations'])
    print('\n', d['file_name'].split('/')[-1])
    for cl in sorted(set(det) | set(gt)):
        print(f'   {cl:40} detecte={det.get(cl,0):>3}  verite={gt.get(cl,0):>3}')
    v = Visualizer(img[:, :, ::-1], metadata=md, scale=1.2)
    cv2_imshow(v.draw_instance_predictions(out).get_image()[:, :, ::-1])

## 6bis. Évaluation par SEUIL — détecté vs vérité par classe
Diagnostic clé : si le rappel remonte en baissant le seuil → **problème de seuil**.
S'il reste plat et `score_max`≈0 pour une classe → **le modèle ne la voit pas** (données insuffisantes).

In [ ]:
from detectron2.engine import DefaultPredictor
from collections import defaultdict
import cv2

THRESHOLDS = [0.05, 0.10, 0.20, 0.30, 0.50]
names = MetadataCatalog.get('plans_val').thing_classes
val = DatasetCatalog.get('plans_val')

cfg.MODEL.WEIGHTS = 'output/model_final.pth'
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.01   # on capte tout, on filtre apres
pred = DefaultPredictor(cfg)

truth  = defaultdict(int)
scores = defaultdict(list)
for d in val:
    for a in d['annotations']:
        truth[a['category_id']] += 1
    inst = pred(cv2.imread(d['file_name']))['instances'].to('cpu')
    for c, s in zip(inst.pred_classes.numpy(), inst.scores.numpy()):
        scores[int(c)].append(float(s))

hdr = f"{'classe':38} {'verite':>6} " + ' '.join(f"{'>='+str(t):>6}" for t in THRESHOLDS)
print(hdr); print('-'*len(hdr))
tt = 0; tot = {t: 0 for t in THRESHOLDS}
for cid in sorted(truth, key=lambda c: -truth[c]):
    row = f"{names[cid][:38]:38} {truth[cid]:>6} "
    for t in THRESHOLDS:
        n = sum(1 for s in scores.get(cid, []) if s >= t)
        row += f"{n:>6} "; tot[t] += n
    tt += truth[cid]; print(row)
print('-'*len(hdr))
print(f"{'TOTAL':38} {tt:>6} " + ' '.join(f"{tot[t]:>6}" for t in THRESHOLDS))
print('\nScore max par classe (voit-il l objet ?):')
for cid in sorted(truth, key=lambda c: -truth[c]):
    ss = scores.get(cid, [])
    print(f"  {names[cid][:38]:38} n_det={len(ss):>3}  score_max={max(ss) if ss else 0:.2f}")

## 7. Sauver le modèle
Télécharge `output/model_final.pth` — c'est le détecteur à utiliser en prod (sur PDF seul, sans DWG).

In [ ]:
# on redescend les POIDS *et* les METRIQUES : sans metrics.json on ne sait rien
# du run une fois l'onglet ferme.
import shutil, os
from google.colab import files
for f in ('output/metrics.json', 'output/coco_instances_results.json'):
    print(f, 'OK' if os.path.exists(f) else 'ABSENT')
shutil.make_archive('run_outputs', 'zip', 'output')
files.download('output/model_final.pth')
files.download('run_outputs.zip')

## 8. Export TorchScript (pour l'inference locale, sans re-upload)

Exporte le modele en **TorchScript** dans CETTE session : le `.pth` est deja sur le
disque Colab (`output/model_final.pth`), donc **aucun upload** de 352 Mo. Produit
`model.ts` + `model_meta.json` a executer en local avec torch-CPU seul.

CPU suffit (pas de GPU requis ici). **A VALIDER** : l'export Mask R-CNN peut demander
une iteration ; le `.pth` est deja telecharge (cellule 7) donc aucune perte si ca casse.
**Copie-colle la sortie de l'introspection** (les formes des sorties) : elle fixe le
decodeur local.

In [ ]:
# EXPORT TORCHSCRIPT -- meme session, .pth deja local, AUCUN upload
import torch, json
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.export import TracingAdapter

# on reconstruit un modele CPU a partir des poids sauves (le trainer.model est sur GPU)
cfg.MODEL.DEVICE = 'cpu'
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05   # bas : le filtrage par classe se fait en local
m = build_model(cfg); m.eval()
DetectionCheckpointer(m).load('output/model_final.pth')

# detectron2 attend list[dict] {'image': CHW BGR 0-255, height, width}. La normalisation
# (PIXEL_MEAN/STD) est faite DANS le modele -> en local, fournir l'image BRUTE BGR 0-255 CHW.
example = torch.rand(3, 800, 800) * 255
inputs = [{'image': example}]  # image seule -> post-traitement en 800x800, ts prend 1 arg
def inference(model, inputs):
    return [{'instances': model(inputs)[0]['instances']}]
adapter = TracingAdapter(m, inputs, inference); adapter.eval()
with torch.no_grad():
    ts = torch.jit.trace(adapter, adapter.flattened_inputs)
ts.save('output/model.ts')
print('model.ts sauve')

# INTROSPECTION des sorties (indispensable pour le decodeur local) -- copie-colle ceci
with torch.no_grad():
    out = ts(torch.rand(3, 800, 800) * 255)
out = out if isinstance(out, (tuple, list)) else (out,)
print(f'{len(out)} sorties :')
for i, t in enumerate(out):
    print(f'  [{i}] shape={tuple(t.shape)} dtype={t.dtype}' if torch.is_tensor(t) else f'  [{i}] {t}')

# noms de classes DANS L'ORDRE des index du modele + format d'entree
names = [c['name'] for c in sorted(
    json.load(open('coco_colab/train/annotations.json'))['categories'], key=lambda c: c['id'])]
json.dump({'classes': names, 'input_size': 800, 'input_format': 'BGR_0_255_CHW'},
          open('output/model_meta.json', 'w'), ensure_ascii=False, indent=1)
files.download('output/model.ts')
files.download('output/model_meta.json')